In [ ]:
# -*- coding: utf-8 -*-
"""數值分析new

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1btUZWbjFN59a4AoiLysxnwZvog8C5CuG

Cell 1｜安裝套件、上傳基金淨值檔
"""

# ===== Cell 1｜安裝套件、上傳基金淨值檔 =====
!pip -q install openpyxl tabulate
import numpy as np, pandas as pd, json, itertools, matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from google.colab import files

up = files.upload()            # 選 active_fund_nav.csv.xlsx
FUND_FILE = next(iter(up))
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
print("已上傳：", FUND_FILE)

### Cell 2｜參數設定與事先寫死的成功標準


In [ ]:
# ===== Cell 2｜參數設定與事先寫死的成功標準 =====
# 注意：修改 EXEC_LAG 後，請從 Cell 4 起重跑
SEED = 42
START = "2013-01-04"                                      # 主要起點
SENS_STARTS = ["2005-01-07", "2013-01-04", "2016-07-01"]  # 多起點敏感度檢查

H_WEEKS, DD_THR = 13, -0.15   # 前瞻 13 週內最大跌幅超過 -15% = 危險
EXEC_LAG = 1                  # 訊號出來後多延遲 1 週生效
COST = 0.002                  # 每單位換手成本（假設值）
RF_ANNUAL = 0.01              # 現金年報酬（假設值）
RF_W = RF_ANNUAL / 52

MIN_TRAIN, TEST_LEN = 208, 52
INNER_MIN_TRAIN, INNER_TEST_LEN = 104, 26

GRID_LO = [round(x, 2) for x in np.arange(0.05, 0.55, 0.05)]
GRID_HI = [round(x, 2) for x in np.arange(0.20, 0.95, 0.10)]
OBJECTIVE, LAMBDA = "cagr_minus_dd", 0.5   # 效用 = 年化報酬 - λ|MDD|；或改 "calmar"

PRIMARY = "logit_all"
BOOT_N, PERM_N, BLOCK = 2000, 1000, 13
CRISIS = {   # 這檔基金自己的大跌段
    "Crash 2015":       ("2015-04-24", "2015-08-28"),
    "Oct 2018":         ("2018-09-28", "2018-11-30"),
    "COVID 2020":       ("2020-01-20", "2020-04-30"),
    "Rate hikes 21-22": ("2021-11-19", "2022-10-28"),
    "Tariff 2025":      ("2025-01-22", "2025-04-30"),
    "Summer 2026":      ("2026-06-22", "2026-08-07"),
}
MIN_WINDOWS_COVERED = 3
MIN_BETTER_FRAC = 0.6
PERM_ALPHA = 0.10
# 成功 = Calmar 差 95% CI 下界 > 0 且 危機區間條件成立 且 置換 p < PERM_ALPHA


### Cell 3｜載入基金淨值、資料稽核、平靜期掃描


In [ ]:
# ===== Cell 3｜載入基金淨值、資料稽核、平靜期掃描 =====
def load_fund(path):
    raw = pd.read_excel(path, header=None)
    for v in raw.iloc[:5, 0].astype(str):
        if "下載日期" in v: print("[基金檔]", v)
    d = pd.to_datetime(raw.iloc[:, 0], format="%Y/%m/%d", errors="coerce")
    v = pd.to_numeric(raw.iloc[:, 1], errors="coerce")
    m = d.notna() & v.notna()
    s = pd.Series(v[m].to_numpy(), index=pd.DatetimeIndex(d[m]), name="FUND").sort_index()
    return s[~s.index.duplicated(keep="last")]

def last_complete_friday(ts):
    ts = ts.normalize()
    return ts - pd.Timedelta(days=(ts.weekday() - 4) if ts.weekday() >= 4 else (ts.weekday() + 3))

fund = load_fund(FUND_FILE)
cut = last_complete_friday(fund.index.max())
price = fund.resample("W-FRI").last().ffill(limit=3).loc[:cut].dropna()

# ---- 稽核 ----
same = fund.diff() == 0
stale = int(same.groupby((~same).cumsum()).sum().max())
gaps = fund.index.to_series().diff().dt.days
audit = pd.DataFrame([dict(first=fund.index.min().date(), last=fund.index.max().date(),
                           n_daily=len(fund), weekly_end=price.index.max().date(),
                           n_weekly=len(price), max_stale_run=stale,
                           max_gap_days=int(gaps.max()))])
audit.to_csv(OUT / "audit.csv", index=False)
print(audit.to_string(index=False))
print("間隔 > 10 天的位置：\n", gaps[gaps > 10].tail(10))
assert fund.index.min() <= pd.Timestamp(START) - pd.Timedelta(weeks=60), "START 之前的歷史不足以計算特徵"

# ---- 平靜期掃描：滾動 26 週年化波動率最低的時段 ----
r_all = price.pct_change()
vol26 = (r_all.rolling(26).std() * np.sqrt(52)).dropna()
ok_end = price.index.max() - pd.Timedelta(weeks=6 * 52)
picked = []
for d, val in vol26[vol26.index <= ok_end].sort_values().items():
    if all(abs((d - x).days) > 180 for x, _ in picked):
        picked.append((d, val))
    if len(picked) >= 8: break
print("\n最平靜的 26 週窗口（window_end 可作為候選起點）：")
print(pd.DataFrame(picked, columns=["window_end", "ann_vol_26w"]).sort_values("window_end").to_string(index=False))
v0 = vol26.loc[:START].iloc[-1]
print(f"\nSTART={START} 前 26 週年化波動率 {v0:.1%}，落在全期 {(vol26 < v0).mean():.0%} 分位")


### Cell 4｜定義特徵、標籤、績效指標與回測函式


In [ ]:
# ===== Cell 4｜定義特徵、標籤、績效指標與回測函式 =====
GROUPS = {
    "ret":   ["ret_4w", "ret_13w", "ret_26w", "ret_52w"],
    "trend": ["dd_13w", "dd_52w", "ma_gap_26w", "ma_gap_52w"],
    "vol":   ["vol_13w", "vol_52w", "vol_ratio", "worst_wk_13w"],
}
ALL_FEATURES = [c for g in GROUPS.values() for c in g]

def make_features(p):
    r = p.pct_change(); f = pd.DataFrame(index=p.index)
    for n in (4, 13, 26, 52): f[f"ret_{n}w"] = p.pct_change(n)
    f["dd_13w"] = p / p.rolling(13).max() - 1
    f["dd_52w"] = p / p.rolling(52).max() - 1
    f["ma_gap_26w"] = p / p.rolling(26).mean() - 1
    f["ma_gap_52w"] = p / p.rolling(52).mean() - 1
    f["vol_13w"] = r.rolling(13).std() * np.sqrt(52)
    f["vol_52w"] = r.rolling(52).std() * np.sqrt(52)
    f["vol_ratio"] = f["vol_13w"] / f["vol_52w"]
    f["worst_wk_13w"] = r.rolling(13).min()
    return f[ALL_FEATURES]

def forward_dd(p, H, lag):
    """t 週出訊號，實際進場價 = p[t+lag]，看其後 H 週內的最大跌幅。"""
    a = p.to_numpy(float); n = len(a); dd = np.full(n, np.nan)
    for t in range(n):
        e = t + lag
        if e + H > n - 1: break
        fut = a[e + 1:e + H + 1]
        if np.isnan(a[e]) or np.isnan(fut).any(): continue
        dd[t] = fut.min() / a[e] - 1
    return pd.Series(dd, index=p.index)

def build_dataset(p, start):
    F = make_features(p).dropna().loc[start:]       # 特徵用起點前的歷史暖機
    ret = p.pct_change().reindex(F.index)
    dd = forward_dd(p, H_WEEKS, EXEC_LAG).reindex(F.index)
    y = (dd < DD_THR).astype(float).where(dd.notna())
    print(f"[建模資料] {F.index.min().date()} ~ {F.index.max().date()}，{len(F)} 週，危險標籤比例 {y.mean():.1%}")
    return F, y, ret

def metrics(r, periods=52):
    r = np.asarray(r, float); eq = np.cumprod(1 + r)
    peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    mdd = float((eq / peak - 1).min()); cagr = float(eq[-1] ** (periods / len(r)) - 1)
    ex = r - RF_W; vol = float(r.std(ddof=1) * np.sqrt(periods))
    dn = float(np.sqrt(np.mean(np.minimum(ex, 0) ** 2)) * np.sqrt(periods))
    return dict(CAGR=cagr, MDD=mdd, Vol=vol,
                Sharpe=float(ex.mean() * periods / vol) if vol > 0 else np.nan,
                Sortino=float(ex.mean() * periods / dn) if dn > 0 else np.nan,
                Calmar=cagr / abs(mdd) if mdd < 0 else np.nan)

def to_position(p, lo, hi):
    return np.clip(1 - (np.asarray(p, float) - lo) / (hi - lo), 0, 1)

def simulate(ret, sig, lag=EXEC_LAG):
    """t 週訊號 -> 第 t+1+lag 週的報酬才生效。含換手成本，現金部位計利息。"""
    pos = sig.shift(1 + lag).fillna(1.0); turn = pos.diff().abs().fillna(0.0)
    return pos * ret + (1 - pos) * RF_W - COST * turn, pos, turn

def objective(r):
    m = metrics(r)
    if OBJECTIVE == "calmar": return m["Calmar"] if np.isfinite(m["Calmar"]) else -np.inf
    return m["CAGR"] - LAMBDA * abs(m["MDD"])


### Cell N1｜共用設定、圖表風格、驗收與發現登記


In [ ]:
# ===== Cell N1｜共用設定：套件、圖表風格、存圖函式、驗收（程式驗證）與發現（實證）登記 =====
import warnings, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import signal, interpolate
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*group delay is singular.*")

FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
COL = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00",
           purple="#CC79A7", sky="#56B4E9", grey="#555555")   # 色盲友善配色
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "axes.prop_cycle": plt.cycler(color=list(COL.values())),
})

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=170, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.svg", bbox_inches="tight")
    plt.show()

# CHECKS：有明確對錯的檢查（kind = "program" 程式/方法驗證，"empirical" 依資料而定的可靠度檢查）
# FINDINGS：實證發現，只記錄數字，不判 PASS/FAIL
CHECKS, FINDINGS, TABLES = {}, {}, {}
def check(name, ok, detail, kind="program"):
    CHECKS[name] = dict(ok=bool(ok), kind=kind, detail=detail)
    print(("PASS  " if ok else "FAIL  ") + f"[{kind}] {name} | {detail}")
def note(name, detail):
    FINDINGS[name] = detail
    print(f"NOTE  {name} | {detail}")

lag_now = lambda lag=None: EXEC_LAG if lag is None else lag   # 呼叫時才讀 EXEC_LAG

EVENTS = {"GFC 2007-09": ("2007-07-01", "2009-03-31"),
          "COVID 2020":  ("2020-01-01", "2020-04-30"),
          "Summer 2026": ("2026-06-01", "2026-08-14")}
print("daily:", fund.index.min().date(), "~", fund.index.max().date(), "| weekly:", len(price), "weeks")


### Cell N2｜模組 A：取樣離散化誤差與外插（F1、F2）


In [ ]:
# ===== Cell N2｜模組 A：MDD 對取樣間隔的收斂與外插（F1）、2026 夏季日對週疊圖（F2） =====
# 注意：基金一天只有一個淨值，實務上有意義的是「日頻 vs 週頻」；h→0 的外插值只是「假設價格為連續過程」下的模型推算值
def mdd_of(x):
    x = np.asarray(x, float); return float((x / np.maximum.accumulate(x) - 1).min())

def mdd_vs_step(s, steps):
    rows = []
    for k in steps:                                    # 所有起始偏移取平均，避免起點偶然性
        v = [mdd_of(s.iloc[o::k]) for o in range(k) if len(s.iloc[o::k]) > 5]
        rows.append((k, np.mean(v), np.std(v)))
    return pd.DataFrame(rows, columns=["step", "mdd", "spread"])

def fit_form(df, p):                                   # MDD(h) = MDD0 + c*h^p
    A = np.c_[np.ones(len(df)), df.step.to_numpy(float) ** p]
    coef = np.linalg.lstsq(A, df.mdd.to_numpy(), rcond=None)[0]
    rms = float(np.sqrt(np.mean((A @ coef - df.mdd.to_numpy()) ** 2)))
    return coef[0], coef[1], rms

A_WIN = {"Full 2000-2026": (None, None, [1, 2, 3, 5, 10, 21]),
         "COVID 2020":     ("2020-01-01", "2020-04-30", [1, 2, 3, 5]),
         "Summer 2026":    ("2026-06-15", "2026-08-14", [1, 2, 3, 5])}

fig, axs = plt.subplots(1, 3, figsize=(16, 4.8)); rows = []
for ax, (nm, (a, b, steps)) in zip(axs, A_WIN.items()):
    s = fund if a is None else fund.loc[a:b]
    wk = price if a is None else price.loc[a:b]
    df = mdd_vs_step(s, steps)
    m1 = df.mdd[df.step == 1].iloc[0]; m2 = df.mdd[df.step == 2].iloc[0]
    M_rich = (2 ** 0.5 * m1 - m2) / (2 ** 0.5 - 1)     # Richardson 外插（假設誤差 ~ sqrt(h)）
    f5, f1 = fit_form(df, 0.5), fit_form(df, 1.0)
    gap, shift = abs(f5[0] - f1[0]), abs(f5[0] - m1)
    ok = gap < shift                                   # 擬合形式間差距 < 外插位移，外插才有辨識力
    rows.append(dict(window=nm, daily_MDD=m1, weekly_Fri_MDD=mdd_of(wk), gap_daily_minus_weekly=m1 - mdd_of(wk),
                     fit_sqrt_h=f5[0], fit_h=f1[0], richardson=M_rich, form_gap=gap, extrap_shift=shift, extrap_reliable=ok))
    check(f"A_extrap_{nm}", ok, f"form gap {gap:.4f} vs extrapolation shift {shift:.4f}", kind="empirical")
    note(f"A_daily_vs_weekly_{nm}", f"daily MDD {m1:.3f}, weekly Fri MDD {mdd_of(wk):.3f}")
    ax.errorbar(df.step, df.mdd, yerr=df.spread, fmt="o", color=COL["blue"], capsize=3, label="sampled MDD")
    hh = np.linspace(0, df.step.max(), 100)
    ax.plot(hh, f5[0] + f5[1] * hh ** 0.5, "--", color=COL["red"], label=f"fit sqrt(h): MDD0={f5[0]:.3f}")
    ax.plot(hh, f1[0] + f1[1] * hh, "--", color=COL["green"], label=f"fit h: MDD0={f1[0]:.3f}")
    ax.scatter([0], [f5[0]], marker="*", s=180, color=COL["red"], zorder=5)
    ax.scatter([0], [f1[0]], marker="*", s=180, color=COL["green"], zorder=5)
    ax.axhline(mdd_of(wk), color=COL["orange"], ls=":", label=f"weekly Fri: {mdd_of(wk):.3f}")
    ax.set_title(nm); ax.set_xlabel("sampling step h (data rows)"); ax.set_ylabel("max drawdown")
    ax.legend(fontsize=8, loc="lower right")
fig.suptitle("F1  Discretization error of max drawdown (h=0 values are model extrapolations)", y=1.03, fontweight="bold")
fig.tight_layout(); save(fig, "F1_mdd_vs_step")
TABLES["A_sampling"] = pd.DataFrame(rows); print(TABLES["A_sampling"].round(4).to_string(index=False))

# ---- F2：2026 夏季，日頻 vs 週頻；註解固定在空白區、圖例移到下方 ----
import matplotlib.dates as mdates
a, b, _ = A_WIN["Summer 2026"]; dd_, ww = fund.loc[a:b], price.loc[a:b]
pk_t, pk_v = dd_.idxmax(), dd_.max(); tr_t, tr_v = dd_.idxmin(), dd_.min()

fig, ax = plt.subplots(figsize=(11, 5.8), layout="constrained")
ax.plot(dd_.index, dd_.values, color=COL["blue"], lw=1.8, label=f"Daily NAV (MDD {mdd_of(dd_):.1%})")
ax.plot(ww.index, ww.values, "o-", color=COL["orange"], lw=1.6, ms=7, label=f"Weekly Friday close (MDD {mdd_of(ww):.1%})")
ax.set_ylim(tr_v * 0.93, pk_v * 1.09)                       # 上下留空間給註解
box = lambda c: dict(boxstyle="round,pad=0.3", fc="white", ec=c, alpha=0.95)
ax.annotate(f"daily peak {pk_v:.1f} ({pk_t:%m/%d})", (pk_t, pk_v), xytext=(0.30, 0.94), textcoords="axes fraction",
            va="top", color=COL["blue"], bbox=box(COL["blue"]), arrowprops=dict(arrowstyle="->", color=COL["blue"]))
ax.annotate(f"daily trough {tr_v:.1f} ({tr_t:%m/%d})", (tr_t, tr_v), xytext=(0.70, 0.08), textcoords="axes fraction",
            va="bottom", color=COL["blue"], bbox=box(COL["blue"]), arrowprops=dict(arrowstyle="->", color=COL["blue"]))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.FR))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.tick_params(axis="x", rotation=45)
ax.set_ylabel("NAV"); ax.set_title("F2  Summer 2026: the weekly view misses part of the fall")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
save(fig, "F2_daily_vs_weekly")


### Cell N3｜模組 B：濾波器頻率響應與延遲對雜訊取捨（F3、F5）Cell N3｜模組 B：頻率響應、群延遲、延遲與趨勢訊雜比（F3、F5）


In [ ]:
# ===== Cell N3｜模組 B：SMA/EMA/區域線性迴歸的頻率響應、群延遲（F3）；延遲 vs 雜訊、趨勢訊雜比（F5） =====
def w_sma(L): return np.ones(L) / L
def w_ema(L):
    a = 2 / (L + 1); w = a * (1 - a) ** np.arange(int(10 * L)); return w / w.sum()
def w_reg(L):                                          # 因果線性迴歸的端點估計（斜坡延遲 = 0）
    k = np.arange(L); kb = k.mean(); return 1 / L - kb * (k - kb) / ((k - kb) ** 2).sum()
def props(w):                                          # (斜坡輸入下的延遲, 白雜訊增益)
    k = np.arange(len(w)); return float((k * w).sum()), float(np.sqrt((w ** 2).sum()))
def rw_noise(w):
    """價格為隨機漫步（每週波動 = 1）時，(價格 - 濾波值) 的雜訊標準差。"""
    c = np.cumsum(w[::-1])[::-1][1:]                   # c_j = sum_{k>=j} w_k, j >= 1
    return float(np.sqrt((c ** 2).sum()))

# ---- 程式驗證：SMA 群延遲 = (L-1)/2 ----
errs = []
for L in (26, 40, 52):
    w = w_sma(L); gd = signal.group_delay((w, [1.0]), w=np.array([1e-3]))[1][0]
    errs += [abs(gd - (L - 1) / 2), abs(props(w)[0] - (L - 1) / 2)]
check("B_group_delay", max(errs) < 0.05, f"max |numeric - (L-1)/2| = {max(errs):.4f} weeks")
check("B_reg_zero_delay", abs(props(w_reg(40))[0]) < 1e-9, f"local linear fit ramp delay = {props(w_reg(40))[0]:.2e}")

# ---- F3：頻率響應與群延遲（在 |H| 很小處遮罩） ----
om = np.linspace(0.01, np.pi, 1500); fr = om / (2 * np.pi)
fig, axs = plt.subplots(1, 2, figsize=(14, 4.8))
for nm, w, c in [("SMA 26", w_sma(26), COL["blue"]), ("SMA 40", w_sma(40), COL["green"]),
                 ("SMA 52", w_sma(52), COL["red"]), ("EMA ~40", w_ema(40), COL["purple"])]:
    h = signal.freqz(w, [1.0], worN=om)[1]
    gd = signal.group_delay((w, [1.0]), w=om)[1]
    gd = np.where(np.abs(h) < 1e-3, np.nan, gd)
    axs[0].plot(fr, 20 * np.log10(np.abs(h) + 1e-12), color=c, label=nm)
    axs[1].plot(fr, gd, color=c, label=nm)
axs[0].set_ylim(-60, 3); axs[0].set_xlabel("frequency (cycles / week)"); axs[0].set_ylabel("gain (dB)")
axs[0].set_title("Low-pass magnitude response"); axs[0].legend()
axs[1].set_ylim(0, 40); axs[1].set_xlabel("frequency (cycles / week)"); axs[1].set_ylabel("group delay (weeks)")
axs[1].set_title("Group delay (masked where |H| < 1e-3)"); axs[1].legend()
fig.suptitle("F3  A moving average is a low-pass filter: longer window = more smoothing, more delay", y=1.03, fontweight="bold")
fig.tight_layout(); save(fig, "F3_filter_response")

# ---- F5：左 = 白雜訊觀點；右 = 隨機漫步下 (價格 - 濾波值) 的趨勢訊雜比 ----
Lg = np.arange(5, 81); rows = []
fig, axs = plt.subplots(1, 2, figsize=(15, 5.5))
for nm, fn, c, ls in [("SMA", w_sma, COL["blue"], "-"), ("EMA (overlaps SMA on left)", w_ema, COL["purple"], "--"),
                      ("Local linear fit", w_reg, COL["green"], "-")]:
    pr = np.array([(*props(fn(L)), rw_noise(fn(L))) for L in Lg])
    axs[0].plot(pr[:, 1], pr[:, 0], ls, color=c, lw=2, label=nm)
    axs[1].plot(Lg, pr[:, 0] / pr[:, 2], ls, color=c, lw=2, label=nm)
    rows += [dict(filter=nm, L=int(L), delay_w=d, white_noise_gain=g, rw_noise=r) for L, (d, g, r) in zip(Lg, pr)]
    if nm == "SMA":
        for L in (26, 40, 52):
            d, g = props(fn(L)); axs[0].scatter([g], [d], s=70, color=c, zorder=5)
            axs[0].annotate(f"L={L}", (g, d), xytext=(8, 4), textcoords="offset points")
axs[0].set_xlabel("white-noise gain sqrt(sum w^2)  (lower = smoother)"); axs[0].set_ylabel("delay on a ramp (weeks)")
axs[0].set_title("Delay vs white-noise gain"); axs[0].legend()
axs[1].set_xlabel("window length L (weeks)"); axs[1].set_ylabel("delay / random-walk noise")
axs[1].set_title("Trend SNR of (price - filter): delay IS the signal"); axs[1].legend()
fig.suptitle("F5  For a price-vs-filter crossover rule, zero delay means zero trend signal", y=1.03, fontweight="bold")
fig.tight_layout(); save(fig, "F5_delay_vs_noise")
TABLES["B_delay_noise"] = pd.DataFrame(rows)
# 右圖縱軸 × (每週趨勢斜率 / 每週波動) = (價格 - 均線) 的期望值 / 雜訊標準差


### Cell N4｜模組 B：三次空頭的訊號日與實際成交日（F4）


In [ ]:
# ===== Cell N4｜三次空頭的出場訊號與實際成交（F4） =====
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

def exit_info(L, a, b, lag=None):
    lag = lag_now(lag)
    ma = price.rolling(L).mean(); w = price.loc[a:b]; pk = w.idxmax()
    seg = price.loc[pk:b] < ma.loc[pk:b]; hit = seg[seg].index
    row = dict(L=L, peak=pk.date(), ramp_delay_ref=(L - 1) / 2, trough_dd=float(w.loc[pk:].min() / w[pk] - 1))
    if len(hit) == 0:
        return {**row, "signal": None, "signal_delay_w": np.nan, "dd_at_signal": np.nan, "exec": None, "dd_at_exec": np.nan}
    t = hit[0]
    te = price.index[min(price.index.get_loc(t) + lag, len(price) - 1)]   # 與回測一致：第 t+lag 週收盤後才離場
    return {**row, "signal": t.date(), "signal_delay_w": (t - pk).days / 7,
            "dd_at_signal": float(price[t] / price[pk] - 1),
            "exec": te.date(), "dd_at_exec": float(price[te] / price[pk] - 1)}

rows = []
fig, axs = plt.subplots(1, 3, figsize=(18, 5.6), layout="constrained")
for i, (ax, (ev, (a, b))) in enumerate(zip(axs, EVENTS.items())):
    a0 = pd.Timestamp(a) - pd.Timedelta(weeks=30)
    sub = price.loc[a0:b]
    ax.plot(sub.index, sub.values, color="black", lw=1.6)
    for L, c in [(26, COL["orange"]), (40, COL["blue"])]:
        ax.plot(sub.index, price.rolling(L).mean().loc[a0:b], color=c, lw=1.4)
        info = exit_info(L, a, b); rows.append({"event": ev, **info})
        if info["signal"] is not None:
            t = pd.Timestamp(info["signal"]); ax.scatter([t], [price[t]], marker="v", s=90, color=c, zorder=5)
        if info["exec"] is not None:
            te = pd.Timestamp(info["exec"]); ax.scatter([te], [price[te]], marker="x", s=70, color=c, zorder=5)
    loc = mdates.AutoDateLocator(minticks=3, maxticks=6)
    ax.xaxis.set_major_locator(loc); ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    ax.set_title(ev, pad=8)
    if i == 0: ax.set_ylabel("weekly NAV")

handles = [Line2D([], [], color="black", lw=1.6), Line2D([], [], color=COL["orange"], lw=1.4),
           Line2D([], [], color=COL["blue"], lw=1.4),
           Line2D([], [], marker="v", ls="", color=COL["grey"], ms=9), Line2D([], [], marker="x", ls="", color=COL["grey"], ms=8)]
labels = ["weekly NAV", "SMA 26", "SMA 40", "exit signal", "execution (signal + lag)"]
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.10), ncol=5)
fig.suptitle("F4  Exit signal (triangle) and execution (x) arrive after the peak", fontweight="bold")
save(fig, "F4_exit_delay")

TABLES["B_exit_delay"] = pd.DataFrame(rows); print(TABLES["B_exit_delay"].round(3).to_string(index=False))
for r in rows:
    note(f"B_exit_{r['event']}_MA{r['L']}",
         f"signal delay {r['signal_delay_w']} w, dd at signal {r['dd_at_signal']}, dd at execution {r['dd_at_exec']}, trough {r['trough_dd']:.3f}")


### Cell N5｜模組 C：曲面、配對 bootstrap、損益兩平成本（F6、F6b、F11）


In [ ]:
# ===== Cell N5｜模組 C：Calmar 曲面（F6）、配對 bootstrap 平原判定與黃金分割（F6b）、Calmar 與 CAGR 損益兩平成本（F11） =====
ret_w = price.pct_change().dropna()
base = ret_w.index[80:]                                # 所有均線長度共用同一評估期，才能互比（與 MA 全期檢定的起點略不同）
Rb = ret_w.loc[base].to_numpy(); Tb = len(Rb)
_sig = {}
def sig_of(L):
    L = int(L)
    if L not in _sig:
        _sig[L] = (price > price.rolling(L).mean()).astype(float).reindex(base).to_numpy()
    return _sig[L]
def strat_r(L, cost, lag=None):
    lag = lag_now(lag); s = sig_of(L); pos = np.r_[np.ones(1 + lag), s[:Tb - 1 - lag]]
    turn = np.abs(np.diff(pos, prepend=pos[0]))
    return pos * Rb + (1 - pos) * RF_W - cost * turn
def calmar_np(r):
    eq = np.cumprod(1 + r); pk = np.maximum.accumulate(np.r_[1.0, eq])[1:]
    mdd = (eq / pk - 1).min(); cagr = eq[-1] ** (52 / len(r)) - 1
    return cagr / abs(mdd) if mdd < 0 else np.nan
def cagr_np(r):
    eq = np.cumprod(1 + r); return eq[-1] ** (52 / len(r)) - 1

CB, GB = calmar_np(Rb), cagr_np(Rb)                    # 買進持有（與成本無關）
Ls = np.arange(10, 81, 2); costs = np.linspace(0, 0.01, 11)
Z = np.array([[calmar_np(strat_r(L, c)) for L in Ls] for c in costs])

# ---- F6：曲面 + 熱圖 ----
X, Y = np.meshgrid(Ls, costs * 100)
gi, gj = np.unravel_index(np.nanargmax(Z), Z.shape)
fig = plt.figure(figsize=(15, 6))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(X, Y, Z, cmap="viridis", alpha=0.92, linewidth=0)
ax1.plot_surface(X, Y, np.full_like(Z, CB), color="grey", alpha=0.25)
ax1.contourf(X, Y, Z, zdir="z", offset=np.nanmin(Z) - 0.1, cmap="viridis", alpha=0.6)
ax1.set_xlabel("MA length (weeks)"); ax1.set_ylabel("cost (%)"); ax1.set_zlabel("Calmar")
ax1.set_title("Calmar surface (grey plane = buy & hold)"); ax1.view_init(26, -58)
ax2 = fig.add_subplot(1, 2, 2)
pm = ax2.pcolormesh(Ls, costs * 100, Z, cmap="viridis", shading="auto"); fig.colorbar(pm, ax=ax2, label="Calmar")
ax2.contour(Ls, costs * 100, Z, levels=[CB], colors="white", linestyles="--", linewidths=2)
ax2.scatter([Ls[gj]], [costs[gi] * 100], marker="*", s=220, color=COL["orange"], edgecolor="black", zorder=5, label="grid best")
ax2.set_xlabel("MA length (weeks)"); ax2.set_ylabel("cost (%)"); ax2.set_title("White dashed line: Calmar = buy & hold"); ax2.legend()
fig.suptitle("F6  Is the optimum a peak or a plateau?", y=1.0, fontweight="bold"); fig.tight_layout(); save(fig, "F6_calmar_surface")

# ---- F6b：配對區塊 bootstrap（所有長度用同一組重抽樣）+ 黃金分割 ----
rg = np.random.default_rng(SEED); BLK = 13; nb = int(np.ceil(Tb / BLK)); B = 300
st = rg.integers(0, Tb - BLK + 1, (B, nb)); bi = (st[:, :, None] + np.arange(BLK)).reshape(B, -1)[:, :Tb]
boot = lambda r: np.array([calmar_np(r[i]) for i in bi])
Lall = np.arange(10, 81); C0 = 0.002
cal = np.array([calmar_np(strat_r(L, C0)) for L in Lall])
bootC = np.array([boot(strat_r(L, C0)) for L in Lall])
se = np.nanstd(bootC, axis=1)
ib = int(np.nanargmax(cal))
se_diff = np.nanstd(bootC - bootC[ib], axis=1)         # 與最佳長度「差值」的標準誤
within_marg = int((cal >= cal[ib] - se[ib]).sum())
within_pair = int((cal >= cal[ib] - se_diff).sum())
bh_se = np.nanstd(boot(Rb))

calls = {}
def f_gs(L):
    Lr = int(round(L))
    if Lr not in calls: calls[Lr] = calmar_np(strat_r(Lr, C0))
    return calls[Lr]
def golden(f, a, b, tol=0.5):
    g = (np.sqrt(5) - 1) / 2; c, d = b - g * (b - a), a + g * (b - a); fc, fd = f(c), f(d)
    while abs(b - a) > tol:
        if fc > fd: b, d, fd = d, c, fc; c = b - g * (b - a); fc = f(c)
        else:       a, c, fc = c, d, fd; d = a + g * (b - a); fd = f(d)
    x = (a + b) / 2; return x, f(x)
xg, fg = golden(f_gs, 10, 80); jg = int(round(xg)) - 10
check("C_golden_vs_grid", fg >= cal[ib] - se_diff[jg],
      f"golden L={int(round(xg))} Calmar={fg:.3f} ({len(calls)} evals) vs grid L={Lall[ib]} {cal[ib]:.3f} "
      f"({len(Lall)} evals); paired SE of gap {se_diff[jg]:.3f}（黃金分割假設單峰，未必成立）")
note("C_plateau", f"lengths within 1 SE of best: marginal {within_marg}/{len(Lall)}, paired {within_pair}/{len(Lall)}; "
                  f"best L={Lall[ib]} Calmar {cal[ib]:.3f}; buy&hold {CB:.3f} (SE {bh_se:.3f})")

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(Lall, cal - se, cal + se, color=COL["blue"], alpha=0.12, label="±1 marginal SE")
ax.fill_between(Lall, cal[ib] - se_diff, cal[ib], color=COL["orange"], alpha=0.25, label="within 1 paired SE of best")
ax.plot(Lall, cal, color=COL["blue"], lw=2, label=f"MA Calmar (cost {C0:.1%})")
ax.axhline(CB, color=COL["grey"], ls="--", label=f"buy & hold (SE {bh_se:.2f})")
ax.scatter(list(calls), list(calls.values()), color=COL["red"], zorder=5, s=40, label=f"golden-section evals ({len(calls)})")
ax.axvline(Lall[ib], color=COL["orange"], ls=":", label=f"grid best L={Lall[ib]}")
ax.set_xlabel("MA length (weeks)"); ax.set_ylabel("Calmar"); ax.legend(fontsize=9)
ax.set_title(f"F6b  {within_pair} of {len(Lall)} lengths are within 1 paired SE of the best"); save(fig, "F6b_plateau")
TABLES["C_calmar_by_L"] = pd.DataFrame(dict(L=Lall, calmar=cal, se=se, se_diff_vs_best=se_diff))

# ---- F11:Calmar 與 CAGR 兩種損益兩平成本（二分法 vs 割線法） ----
def bisect(f, a, b, tol=1e-8):
    fa, fb = f(a), f(b)
    if fa * fb > 0: return None, 0
    n = 0
    while b - a > tol:
        m = (a + b) / 2; fm = f(m); n += 1
        if fa * fm <= 0: b, fb = m, fm
        else: a, fa = m, fm
    return (a + b) / 2, n
def secant(f, x0, x1, lo=0.0, hi=0.05, tol=1e-10, maxit=60):
    f0, f1 = f(x0), f(x1)
    for n in range(1, maxit + 1):
        if f1 == f0: return np.nan, n
        x2 = x1 - f1 * (x1 - x0) / (f1 - f0)
        if not (lo - 0.01 <= x2 <= hi + 0.01): return np.nan, n      # 跑出區間 -> 視為發散
        x0, f0, x1 = x1, f1, x2; f1 = f(x1)
        if abs(x1 - x0) < tol: return x1, n
    return x1, maxit

cg = np.linspace(0, 0.05, 101); rows = []; ok_all = True
ypos = {26: 0.95, 40: 0.86, 52: 0.77}                          # 每條線的標籤放不同高度
no_root = {0: [], 1: []}
box = lambda c: dict(boxstyle="round,pad=0.25", fc="white", ec=c, alpha=0.95)
fig, axs = plt.subplots(1, 2, figsize=(16, 6), layout="constrained")
for L, c in [(26, COL["orange"]), (40, COL["blue"]), (52, COL["green"])]:
    for j, (nm, fn, ref) in enumerate([("Calmar", calmar_np, CB), ("CAGR", cagr_np, GB)]):
        f = lambda cc, L=L, fn=fn, ref=ref: fn(strat_r(L, cc)) - ref
        axs[j].plot(cg * 100, [f(cc) + ref for cc in cg], color=c, lw=2, label=f"MA{L}")
        xb, nbis = bisect(f, 0.0, 0.05); xs, nsec = secant(f, 0.01, 0.03)
        agree = (xb is None) or (np.isfinite(xs) and abs(xb - xs) < 1e-4)
        ok_all &= agree
        if xb is None:
            status = "below B&H even at zero cost" if f(0.0) < 0 else "above B&H up to 5% cost"
            no_root[j].append(f"MA{L}: {status}")
        else:
            status = ""
            axs[j].axvline(xb * 100, color=c, ls=":", lw=1.2)
            axs[j].scatter([xb * 100], [ref], s=80, color=c, edgecolor="black", zorder=5)
            axs[j].text(xb * 100 + 0.08, ypos[L], f"MA{L}: {xb:.2%}", transform=axs[j].get_xaxis_transform(),
                        color=c, fontsize=10, va="top", bbox=box(c))
        rows.append(dict(L=L, metric=nm, breakeven_bisect=xb, iters_bisect=nbis,
                         breakeven_secant=xs, iters_secant=nsec, note=status))
for j, (nm, ref) in enumerate([("Calmar", CB), ("CAGR", GB)]):
    axs[j].axhline(ref, color=COL["grey"], ls="--", label="buy & hold")
    axs[j].set_xlabel("cost per unit turnover (%)"); axs[j].set_ylabel(nm); axs[j].set_title(f"Break-even on {nm}", pad=8)
    if no_root[j]:
        axs[j].text(0.02, 0.04, "No break-even in [0, 5%]:\n" + "\n".join(no_root[j]), transform=axs[j].transAxes,
                    va="bottom", fontsize=9, bbox=box(COL["grey"]))
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=4)
fig.suptitle("F11  Break-even cost: risk-adjusted vs raw return tell different stories", fontweight="bold")
save(fig, "F11_breakeven_cost")

TABLES["C_breakeven"] = pd.DataFrame(rows); print(TABLES["C_breakeven"].to_string(index=False))
check("C_root_methods_agree", ok_all, "有根時二分法與割線法相差 < 1e-4；無根不算失敗，另在表中註記")
for r in rows:
    note(f"C_breakeven_MA{r['L']}_{r['metric']}", f"{r['breakeven_bisect']} {r['note']}")


### Cell N6｜模組 D1：SDE 數值方法的收斂階（F8）


In [ ]:
# ===== Cell N6｜模組 D1：Euler-Maruyama 與 Milstein 的強/弱收斂階（F8）；參數已在執行前改成雜訊項主導 =====
def sde_convergence(mu=2.0, sig=1.0, T=1.0, N=5000, kmax=11, ks=range(5, 10), seed=SEED):
    rg = np.random.default_rng(seed); nf = 2 ** kmax
    dW = rg.normal(0, np.sqrt(T / nf), (N, nf))
    S_ex = np.exp((mu - 0.5 * sig ** 2) * T + sig * dW.sum(1))      # GBM 精確解（S0=1）
    rows = []
    for k in ks:
        n = 2 ** k; dt = T / n; dWc = dW.reshape(N, n, nf // n).sum(2)
        Se = np.ones(N); Sm = np.ones(N)
        for j in range(n):
            d = dWc[:, j]
            Se = Se * (1 + mu * dt + sig * d)                                   # Euler-Maruyama
            Sm = Sm * (1 + mu * dt + sig * d + 0.5 * sig ** 2 * (d ** 2 - dt))  # Milstein
        rows.append(dict(dt=dt, strong_EM=np.mean(np.abs(S_ex - Se)), strong_Mil=np.mean(np.abs(S_ex - Sm)),
                         weak_EM=abs(np.mean(Se - S_ex)), weak_Mil=abs(np.mean(Sm - S_ex))))   # 同路徑差值降低雜訊
    return pd.DataFrame(rows)

conv = sde_convergence(); TABLES["D_sde_convergence"] = conv
slope = {c: np.polyfit(np.log(conv.dt), np.log(conv[c]), 1)[0] for c in conv.columns if c != "dt"}
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, (kind, cols) in zip(axs, [("Strong error  E|S - S_h|", ["strong_EM", "strong_Mil"]),
                                  ("Weak error  |E[S] - E[S_h]|", ["weak_EM", "weak_Mil"])]):
    for c, col, nm in zip(cols, [COL["red"], COL["blue"]], ["Euler-Maruyama", "Milstein"]):
        ax.loglog(conv.dt, conv[c], "o-", color=col, lw=2, label=f"{nm}  slope = {slope[c]:.2f}")
    for p_, ls in [(0.5, ":"), (1.0, "--")]:
        ax.loglog(conv.dt, conv[cols[0]].iloc[0] * (conv.dt / conv.dt.iloc[0]) ** p_, ls, color=COL["grey"], label=f"reference slope {p_}")
    ax.set_xlabel("step size dt"); ax.set_ylabel("error"); ax.set_title(kind); ax.legend()
fig.suptitle("F8  Measured convergence orders of SDE schemes (GBM, exact solution known)", y=1.02, fontweight="bold")
fig.tight_layout(); save(fig, "F8_sde_convergence")
check("D_EM_strong", abs(slope["strong_EM"] - 0.5) < 0.15, f"slope {slope['strong_EM']:.2f} (theory 0.5)")
check("D_Mil_strong", abs(slope["strong_Mil"] - 1.0) < 0.15, f"slope {slope['strong_Mil']:.2f} (theory 1.0)")
check("D_weak_orders", all(abs(slope[c] - 1.0) < 0.15 for c in ("weak_EM", "weak_Mil")),
      f"EM {slope['weak_EM']:.2f}, Milstein {slope['weak_Mil']:.2f} (theory 1.0)")


### Cell N7｜模組 D2：蒙地卡羅、相同曝險基準、變異數縮減（F9、F10、F13）


In [ ]:
# ===== Cell N7｜模組 D2：校準（同源，非獨立驗證）、GBM 與兩狀態模擬、相同曝險固定比例基準、扇形圖、小提琴圖、MC 收斂與對偶變數 =====
# ---- 校準：週對數報酬；狀態用「上週為止」的 13 週波動率判定 ----
lr = np.log(price).diff().dropna(); mu_g, sg_g = float(lr.mean()), float(lr.std())
v13 = lr.rolling(13).std().shift(1).dropna()
stt = (v13 > v13.quantile(0.75)).astype(int); lrs = lr.loc[stt.index]
mu_k = np.array([lrs[stt == k].mean() for k in (0, 1)]); sg_k = np.array([lrs[stt == k].std() for k in (0, 1)])
cnt = np.zeros((2, 2))
for a_, b_ in zip(stt.values[:-1], stt.values[1:]): cnt[a_, b_] += 1
Pm = cnt / cnt.sum(1, keepdims=True); pi_ = np.linalg.matrix_power(Pm, 500)[0]
TABLES["D_calibration"] = pd.DataFrame(dict(state=["calm", "stress"], weekly_mu=mu_k, weekly_sigma=sg_k,
                                            stay_prob=np.diag(Pm), stationary=pi_))
print(TABLES["D_calibration"].round(4).to_string(index=False)); print("GBM weekly mu, sigma:", round(mu_g, 5), round(sg_g, 4))

WARM = 80
def sim_gbm(N, T, rg, anti=False):
    n = N // 2 if anti else N; z = rg.standard_normal((n, T))
    if anti: z = np.vstack([z, -z])
    return mu_g + sg_g * z                                # 對數報酬；週步長為精確抽樣，無離散化誤差
def sim_regime(N, T, rg):
    s = (rg.random(N) < pi_[1]).astype(int); out = np.empty((N, T)); z = rg.standard_normal((N, T)); u = rg.random((N, T))
    for t in range(T):
        out[:, t] = mu_k[s] + sg_k[s] * z[:, t]; s = (u[:, t] < Pm[s, 1]).astype(int)
    return out
def path_metrics(r):
    eq = np.cumprod(1 + r, axis=1); pk = np.maximum(np.maximum.accumulate(eq, axis=1), 1.0)
    mdd = (eq / pk - 1).min(axis=1); cagr = eq[:, -1] ** (52 / r.shape[1]) - 1
    return dict(mdd=mdd, cagr=cagr, calmar=cagr / np.abs(np.minimum(mdd, -1e-9)))
def eval_paths(lrp, Ls_=(26, 40), cost=None, lag=None, warm=WARM):
    cost = COST if cost is None else cost; lag = lag_now(lag)
    N, M = lrp.shape; p = np.exp(np.cumsum(lrp, axis=1))
    cs = np.concatenate([np.zeros((N, 1)), np.cumsum(p, axis=1)], axis=1)
    tt = np.arange(warm, M); R = p[:, tt] / p[:, tt - 1] - 1; u = tt - 1 - lag
    out = {"BuyHold": path_metrics(R)}
    for L in Ls_:
        sig = (p[:, u] > (cs[:, u + 1] - cs[:, u + 1 - L]) / L).astype(float)
        turn = np.abs(np.diff(sig, axis=1, prepend=sig[:, :1]))
        out[f"MA{L}"] = path_metrics(sig * R + (1 - sig) * RF_W - cost * turn)
        e = sig.mean(axis=1, keepdims=True)                       # 該路徑上均線規則的平均曝險
        out[f"Const{L}"] = path_metrics(e * R + (1 - e) * RF_W)   # 相同曝險固定比例（每週再平衡、未計成本，對它略有利）
    return out, p

# ---- 主模擬 ----
rg = np.random.default_rng(SEED); N_MC = 2000; T_MC = Tb
res_g, _ = eval_paths(sim_gbm(N_MC, T_MC + WARM, rg))
res_r, p_r = eval_paths(sim_regime(N_MC, T_MC + WARM, rg))
hist = {"BuyHold": path_metrics(Rb[None, :]), "MA26": path_metrics(strat_r(26, COST)[None, :]),
        "MA40": path_metrics(strat_r(40, COST)[None, :])}
rows = []
for sc, res in [("GBM", res_g), ("Regime", res_r)]:
    for k, v in res.items():
        row = dict(scenario=sc, strat=k, mdd_median=np.median(v["mdd"]), mdd_p05=np.percentile(v["mdd"], 5),
                   cagr_median=np.median(v["cagr"]), calmar_median=np.median(v["calmar"]))
        if k.startswith("MA"):
            mt = res["Const" + k[2:]]
            row.update(P_mdd_better_than_BH=float(np.mean(v["mdd"] > res["BuyHold"]["mdd"])),
                       P_mdd_better_than_matched=float(np.mean(v["mdd"] > mt["mdd"])),
                       P_calmar_better_than_matched=float(np.mean(v["calmar"] > mt["calmar"])))
            note(f"D_{sc}_{k}", f"P(MDD better than B&H) {row['P_mdd_better_than_BH']:.2f}, "
                                f"P(MDD better than matched) {row['P_mdd_better_than_matched']:.2f}, "
                                f"P(Calmar better than matched) {row['P_calmar_better_than_matched']:.2f}")
        rows.append(row)
for k, v in hist.items():
    rows.append(dict(scenario="History", strat=k, mdd_median=v["mdd"][0], cagr_median=v["cagr"][0], calmar_median=v["calmar"][0]))
TABLES["D_mc_summary"] = pd.DataFrame(rows); print(TABLES["D_mc_summary"].round(3).to_string(index=False))

# ---- F9：扇形圖（模型以同一條路徑校準，吻合不是驗證） ----
pp = p_r[:, WARM - 1:] / p_r[:, [WARM - 1]]; q = np.percentile(pp, [5, 25, 50, 75, 95], axis=0); x = np.arange(pp.shape[1])
hp = price.loc[ret_w.index[79]:]; hp = (hp / hp.iloc[0]).to_numpy()
fig, ax = plt.subplots(figsize=(11, 6))
for i in range(25): ax.plot(x, pp[i], color=COL["sky"], lw=0.5, alpha=0.5)
ax.fill_between(x, q[0], q[4], color=COL["blue"], alpha=0.15, label="5-95%")
ax.fill_between(x, q[1], q[3], color=COL["blue"], alpha=0.30, label="25-75%")
ax.plot(x, q[2], color=COL["blue"], lw=2, label="median")
ax.plot(np.arange(len(hp)), hp, color="black", lw=1.8, label="actual history (used for calibration)"); ax.set_yscale("log")
ax.set_xlabel("weeks"); ax.set_ylabel("normalized NAV (log)"); ax.legend()
ax.set_title("F9  Two-state regime model: simulated NAV fan (scenario analysis, not validation)"); save(fig, "F9_fan_chart")

# ---- F10：MDD 分布，含相同曝險基準 ----
fig, ax = plt.subplots(figsize=(13, 5.5)); labels, pos_ = [], 0
cols = {"BuyHold": COL["grey"], "Const40": COL["green"], "MA26": COL["orange"], "MA40": COL["blue"]}
for sc, res in [("GBM", res_g), ("Regime", res_r)]:
    for k in ("BuyHold", "Const40", "MA26", "MA40"):
        pos_ += 1; vp = ax.violinplot(res[k]["mdd"], positions=[pos_], showmedians=True, widths=0.8)
        for b in vp["bodies"]: b.set_facecolor(cols[k]); b.set_alpha(0.55)
        if k in hist:
            ax.scatter([pos_], [hist[k]["mdd"][0]], marker="D", s=70, color="black", zorder=5,
                       label="actual history" if pos_ == 1 else None)
        labels.append(f"{sc}\n{k}")
ax.set_xticks(range(1, pos_ + 1)); ax.set_xticklabels(labels); ax.set_ylabel("max drawdown"); ax.legend()
ax.set_title("F10  MDD under simulated paths: compare MA with the exposure-matched constant mix (Const40)")
save(fig, "F10_mdd_violin")

# ---- F13：蒙地卡羅誤差 ~ 1/sqrt(N)，以及對偶變數 ----
def est(N, rg, anti=False, T=260):
    return eval_paths(sim_gbm(N, T + WARM, rg, anti), Ls_=(40,))[0]["MA40"]["mdd"].mean()
Ns = [125, 250, 500, 1000, 2000, 4000]; REP = 30; rg = np.random.default_rng(SEED + 1)
sd = [np.std([est(N, rg) for _ in range(REP)], ddof=1) for N in Ns]
sl = np.polyfit(np.log(Ns), np.log(sd), 1)[0]
v_plain = np.var([est(1000, rg) for _ in range(60)], ddof=1); v_anti = np.var([est(1000, rg, True) for _ in range(60)], ddof=1)
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.loglog(Ns, sd, "o-", color=COL["blue"], lw=2, label=f"plain MC, slope = {sl:.2f}")
ax.loglog(Ns, sd[0] * (np.array(Ns) / Ns[0]) ** -0.5, "--", color=COL["grey"], label="reference slope -0.5")
ax.scatter([1000], [np.sqrt(v_anti)], marker="*", s=220, color=COL["red"], zorder=5,
           label=f"antithetic (N=1000): variance ratio {v_plain / v_anti:.2f}")
ax.set_xlabel("number of paths N"); ax.set_ylabel("std of the estimated mean MDD"); ax.legend()
ax.set_title("F13  Monte Carlo error decays like 1/sqrt(N)"); save(fig, "F13_mc_convergence")
check("D_mc_rate", abs(sl + 0.5) < 0.1, f"slope {sl:.2f} (theory -0.5)")
TABLES["D_variance_reduction"] = pd.DataFrame([dict(var_plain=v_plain, var_antithetic=v_anti, ratio=v_plain / v_anti)])
note("D_antithetic", f"variance ratio plain/antithetic = {v_plain / v_anti:.2f}（<=1 表示沒有幫助，如實報告）")


### Cell N8｜模組 C2：條件數、SVD、牛頓法對梯度下降（F7a、F7b）


In [ ]:
# ===== Cell N8｜模組 C2：12 個價格特徵的共線性、SVD、條件數（F7a）；Logistic 的牛頓法 vs 梯度下降（F7b） =====
D_e, y_e, _ = build_dataset(price, "2005-01-07")
m = y_e.notna(); Xdf = D_e.loc[m, ALL_FEATURES]; yv = y_e[m].to_numpy()
Xs = ((Xdf - Xdf.mean()) / Xdf.std()).to_numpy(); n, d = Xs.shape
sv = np.linalg.svd(Xs, compute_uv=False); condX = sv.max() / sv.min()
lams = np.logspace(-6, 1, 30)
cond_l = [((sv.max() ** 2 / n) + l) / ((sv.min() ** 2 / n) + l) for l in lams]

fig, axs = plt.subplots(1, 3, figsize=(17, 5))
cm = axs[0].pcolormesh(Xdf.corr().to_numpy(), cmap="coolwarm", vmin=-1, vmax=1); fig.colorbar(cm, ax=axs[0])
axs[0].set_xticks(np.arange(d) + 0.5); axs[0].set_xticklabels(ALL_FEATURES, rotation=90, fontsize=7)
axs[0].set_yticks(np.arange(d) + 0.5); axs[0].set_yticklabels(ALL_FEATURES, fontsize=7)
axs[0].set_title("Feature correlation"); axs[0].grid(False)
axs[1].semilogy(np.arange(1, d + 1), sv, "o-", color=COL["blue"], lw=2)
axs[1].set_xlabel("component"); axs[1].set_ylabel("singular value"); axs[1].set_title(f"SVD scree  (cond = {condX:.1f})")
axs[2].loglog(lams, cond_l, color=COL["red"], lw=2); axs[2].set_xlabel("ridge lambda"); axs[2].set_ylabel("cond(X'X/n + lambda I)")
axs[2].set_title("Ridge tames the condition number")
fig.suptitle("F7a  The 12 price features are strongly collinear", y=1.02, fontweight="bold"); fig.tight_layout(); save(fig, "F7a_conditioning")
note("C2_condition_number", f"cond(X) = {condX:.1f}")

# ---- 牛頓法（IRLS） ----
Xa = np.c_[np.ones(n), Xs]; lam = 1e-2; pen = np.r_[0.0, np.ones(d)]
sigm = lambda z: 1 / (1 + np.exp(-np.clip(z, -30, 30)))
w = np.zeros(d + 1); ws = [w.copy()]
for _ in range(40):
    p = sigm(Xa @ w); g = Xa.T @ (p - yv) / n + lam * pen * w
    H = (Xa * (p * (1 - p))[:, None]).T @ Xa / n + np.diag(lam * pen + 1e-12)
    step = np.linalg.solve(H, g); w = w - step; ws.append(w.copy())
    if np.linalg.norm(step) < 1e-14: break
ws = np.array(ws); w_star = ws[-1]; e_newton = np.linalg.norm(ws - w_star, axis=1)[:-1]; condH = np.linalg.cond(H)

# ---- 梯度下降（步長 1/Lipschitz） ----
Lip = 0.25 * np.linalg.eigvalsh(Xa.T @ Xa / n).max() + lam
w = np.zeros(d + 1); e_gd = [np.linalg.norm(w - w_star)]
for _ in range(3000):
    p = sigm(Xa @ w); w = w - (Xa.T @ (p - yv) / n + lam * pen * w) / Lip; e_gd.append(np.linalg.norm(w - w_star))
e_gd = np.array(e_gd); it6 = int(np.argmax(e_gd < 1e-6)) if (e_gd < 1e-6).any() else None

fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.semilogy(np.arange(len(e_newton)), np.maximum(e_newton, 1e-17), "o-", color=COL["red"], lw=2, ms=8,
            label=f"Newton / IRLS ({len(e_newton)} iterations)")
ax.semilogy(np.arange(len(e_gd)), e_gd, color=COL["blue"], lw=2, label=f"Gradient descent (1e-6 at iteration {it6})")
ax.set_xlim(-1, 400); ax.set_xlabel("iteration"); ax.set_ylabel("||w_k - w*||"); ax.legend()
ax.set_title(f"F7b  Quadratic vs linear convergence  (cond(Hessian) = {condH:.0f})"); save(fig, "F7b_newton_vs_gd")

# ---- 超線性收斂檢查：誤差比值是否快速趨近 0（線性收斂時比值維持常數） ----
ee = e_newton[e_newton > 1e-13]
rat = ee[1:] / ee[:-1]; qq = ee[1:] / ee[:-1] ** 2
ok = len(rat) >= 2 and rat[-1] < 1e-2 * rat[0]
check("C2_newton_superlinear", ok,
      f"error ratios {np.round(rat, 6).tolist()}；e_(k+1)/e_k^2 = {np.round(qq, 3).tolist()}（二次收斂時應大致有界）")
gd_tail = e_gd[200:400]; gd_ratio = float(np.exp(np.mean(np.diff(np.log(gd_tail))))) if (gd_tail > 0).all() else np.nan
note("C2_gd_linear_rate", f"gradient descent mean error ratio per iteration ≈ {gd_ratio:.4f}（接近常數 = 線性收斂）")
TABLES["C2_conditioning"] = pd.DataFrame([dict(n_samples=n, cond_X=condX, cond_Hessian=condH,
                                               newton_iters=len(e_newton), gd_iters_to_1e6=it6, gd_ratio=gd_ratio)])


### Cell N9｜模組 E：插值誤差與 Runge 現象（F12）


In [ ]:
# ===== Cell N9｜模組 E：線性/三次樣條/PCHIP 插值誤差階數，以及 Runge 現象（F12） =====
y_lp = np.log(price).to_numpy(); xg_ = np.arange(len(y_lp))
def interp_err(step):
    xs = np.arange(0, len(y_lp), step); xs = xs if xs[-1] == len(y_lp) - 1 else np.r_[xs, len(y_lp) - 1]
    cand = {"linear": np.interp(xg_, xs, y_lp[xs]), "cubic spline": interpolate.CubicSpline(xs, y_lp[xs])(xg_),
            "PCHIP": interpolate.PchipInterpolator(xs, y_lp[xs])(xg_)}
    msk = ~np.isin(xg_, xs)
    return {k: float(np.sqrt(np.mean((v - y_lp)[msk] ** 2))) for k, v in cand.items()}
steps = [2, 4, 8, 16]; E_price = pd.DataFrame([interp_err(s) for s in steps], index=steps)

hs = [1 / 8, 1 / 16, 1 / 32, 1 / 64]; xf = np.linspace(0, 1, 4001); f = lambda t: np.sin(2 * np.pi * t); rows = []
for h in hs:
    xs = np.linspace(0, 1, int(round(1 / h)) + 1)
    rows.append(dict(linear=np.sqrt(np.mean((np.interp(xf, xs, f(xs)) - f(xf)) ** 2)),
                     cubic=np.sqrt(np.mean((interpolate.CubicSpline(xs, f(xs))(xf) - f(xf)) ** 2))))
E_smooth = pd.DataFrame(rows, index=hs)
sl_p = {c: np.polyfit(np.log(steps), np.log(E_price[c]), 1)[0] for c in E_price.columns}
sl_s = {c: np.polyfit(np.log(hs), np.log(E_smooth[c]), 1)[0] for c in E_smooth.columns}
check("E_smooth_orders", 1.8 < sl_s["linear"] < 2.2 and 3.5 < sl_s["cubic"] < 4.5,
      f"linear {sl_s['linear']:.2f} (theory 2), cubic {sl_s['cubic']:.2f} (theory 4)")
note("E_price_path_slopes", f"{ {k: round(v, 2) for k, v in sl_p.items()} }（假設：粗糙的價格路徑上各方法都約 0.5）")

fig, axs = plt.subplots(1, 3, figsize=(17, 5))
for c, col in zip(E_price.columns, [COL["blue"], COL["red"], COL["green"]]):
    axs[0].loglog(steps, E_price[c], "o-", color=col, lw=2, label=f"{c}  slope {sl_p[c]:.2f}")
axs[0].loglog(steps, E_price.iloc[0, 0] * (np.array(steps) / steps[0]) ** 0.5, ":", color=COL["grey"], label="slope 0.5")
axs[0].set_xlabel("node spacing (weeks)"); axs[0].set_ylabel("RMSE of log NAV"); axs[0].set_title("Real NAV path: rough data")
axs[0].legend(fontsize=9)
for c, col in zip(E_smooth.columns, [COL["blue"], COL["red"]]):
    axs[1].loglog(hs, E_smooth[c], "o-", color=col, lw=2, label=f"{c}  slope {sl_s[c]:.2f}")
axs[1].set_xlabel("node spacing h"); axs[1].set_ylabel("RMSE"); axs[1].set_title("Smooth test function sin(2 pi x)"); axs[1].legend()
t = np.linspace(-1, 1, 801); fr_ = lambda x: 1 / (1 + 25 * x ** 2); nn = 13
xe = np.linspace(-1, 1, nn); xc = np.cos((2 * np.arange(nn) + 1) * np.pi / (2 * nn))
axs[2].plot(t, fr_(t), "k", lw=2, label="f(x) = 1/(1+25x^2)")
axs[2].plot(t, interpolate.BarycentricInterpolator(xe, fr_(xe))(t), color=COL["red"], label="degree 12, equispaced nodes")
axs[2].plot(t, interpolate.BarycentricInterpolator(xc, fr_(xc))(t), color=COL["green"], label="degree 12, Chebyshev nodes")
axs[2].set_ylim(-1, 1.6); axs[2].set_title("Runge phenomenon"); axs[2].legend(fontsize=9)
fig.suptitle("F12  Interpolation: textbook orders appear on smooth data, not on a price path", y=1.02, fontweight="bold")
fig.tight_layout(); save(fig, "F12_interpolation")
TABLES["E_price_rmse"] = E_price; TABLES["E_smooth_rmse"] = E_smooth


### Cell N10｜驗收：程式驗證與實證發現分開列


In [ ]:
# ===== Cell N10｜驗收：程式驗證、資料可靠度檢查、實證發現分開列出，不合計通過率 =====
def show(kind, title):
    items = {k: v for k, v in CHECKS.items() if v["kind"] == kind}
    print(f"\n=== {title} ===")
    for k, v in items.items():
        print(f"{'PASS' if v['ok'] else 'FAIL'}  {k:28s} {v['detail']}")
    if items: print(f"-> {sum(v['ok'] for v in items.values())} / {len(items)}")
show("program", "程式與數值方法驗證（通過只代表程式正確，不代表策略有效）")
show("empirical", "依資料而定的可靠度檢查")
print("\n=== 實證發現（只記錄，不判對錯） ===")
for k, v in FINDINGS.items(): print(f"- {k}: {v}")
print("\nFAIL 項目要如實寫進報告，不要事後修改標準或參數讓它通過。")


### Cell N11｜存表並打包下載


In [ ]:
# ===== Cell N11｜把表格、驗收與發現存檔，打包 outputs（含 figures）下載 =====
for k, v in TABLES.items():
    v.to_csv(OUT / f"num_{k}.csv", index=not isinstance(v.index, pd.RangeIndex))
(OUT / "num_checks.json").write_text(json.dumps(CHECKS, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
(OUT / "num_findings.json").write_text(json.dumps(FINDINGS, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
!zip -qr outputs_numerics.zip outputs
files.download("outputs_numerics.zip")

# ===== 補充檢核 =====
# 1) 平均曝險(說明書稱「約七成」)
def mean_exposure(lrp, L, lag=EXEC_LAG, warm=WARM):
    N, M = lrp.shape; p = np.exp(np.cumsum(lrp, axis=1))
    cs = np.concatenate([np.zeros((N, 1)), np.cumsum(p, axis=1)], axis=1)
    tt = np.arange(warm, M); u = tt - 1 - lag
    return (p[:, u] > (cs[:, u + 1] - cs[:, u + 1 - L]) / L).mean(axis=1)

rg_x = np.random.default_rng(SEED)                  # 與主模擬相同的抽樣順序,可重現同一批路徑
lr_g = sim_gbm(N_MC, T_MC + WARM, rg_x); lr_r = sim_regime(N_MC, T_MC + WARM, rg_x)
for L in (26, 40):
    print(f"MA{L} 歷史平均曝險 {sig_of(L).mean():.1%} | "
          f"GBM 中位數 {np.median(mean_exposure(lr_g, L)):.1%} | "
          f"Regime 中位數 {np.median(mean_exposure(lr_r, L)):.1%}")

# 2) 最長停滯段發生在哪裡
same = fund.diff() == 0; grp = (~same).cumsum()
g = same.groupby(grp).sum().idxmax()
print("\n最長停滯段:\n", fund[grp == g])

# 3) Runge:把看圖改成數值判定
t_ = np.linspace(-1, 1, 2001); fr_ = lambda x: 1 / (1 + 25 * x ** 2); nn = 13
xe = np.linspace(-1, 1, nn); xc = np.cos((2 * np.arange(nn) + 1) * np.pi / (2 * nn))
err_e = np.abs(interpolate.BarycentricInterpolator(xe, fr_(xe))(t_) - fr_(t_)).max()
err_c = np.abs(interpolate.BarycentricInterpolator(xc, fr_(xc))(t_) - fr_(t_)).max()
print(f"\nRunge 最大誤差:等距 {err_e:.3f},Chebyshev {err_c:.3f}")

# 4) F9:真實歷史終值落在模擬分布的哪個分位
pp_end = p_r[:, -1] / p_r[:, WARM - 1]
hp_ = price.loc[ret_w.index[79]:]; h_end = hp_.iloc[-1] / hp_.iloc[0]
print(f"真實終值 {h_end:.2f},位於模擬分布第 {(pp_end < h_end).mean():.0%} 分位")

for sc, res in [("GBM", res_g), ("Regime", res_r)]:
    print(sc, f"歷史 MA26 Calmar 0.752 位於第 {np.mean(res['MA26']['calmar'] < 0.752):.0%} 分位")

# ===== 選擇性延伸 E1–E9(不改動原有結果) =====
def dd_info(r, idx):
    eq = np.cumprod(1 + r); pk = np.maximum.accumulate(np.r_[1.0, eq])[1:]
    dd = eq / pk - 1; i = int(dd.argmin()); j = int(np.argmax(eq[:i + 1])) if i > 0 else 0
    return float(dd[i]), idx[j].date(), idx[i].date()

# ---- E1 罕見程度:確切路徑數 + CAGR / MDD 拆解 ----
h = hist["MA26"]
for sc, res in [("GBM", res_g), ("Regime", res_r)]:
    m = res["MA26"]
    print(f"[E1] {sc}: Calmar≥{h['calmar'][0]:.3f} 路徑 {int((m['calmar'] >= h['calmar'][0]).sum())}/{len(m['calmar'])} | "
          f"CAGR 分位 {np.mean(m['cagr'] < h['cagr'][0]):.1%} | MDD 分位 {np.mean(m['mdd'] < h['mdd'][0]):.1%}")

# ---- E2 歷史的相同曝險固定比例,在模擬中是否也罕見 ----
e26 = sig_of(26).mean(); c_const = calmar_np(e26 * Rb + (1 - e26) * RF_W)
for sc, res in [("GBM", res_g), ("Regime", res_r)]:
    print(f"[E2] {sc}: 歷史 Const26(曝險 {e26:.1%})Calmar {c_const:.3f},位於第 {np.mean(res['Const26']['calmar'] < c_const):.1%} 分位")

# ---- E3 動能檢查:變異數比 VR(q) 與一階自相關,歷史 vs 模擬 ----
def vr(x, q):
    x = np.atleast_2d(x); x = x - x.mean(1, keepdims=True)
    cs = np.cumsum(x, 1); s = cs[:, q:] - cs[:, :-q]
    return s.var(1) / (q * x.var(1))
def ac1(x):
    x = np.atleast_2d(x); x = x - x.mean(1, keepdims=True)
    return (x[:, 1:] * x[:, :-1]).mean(1) / x.var(1)
rg_x = np.random.default_rng(SEED)                       # 與主模擬相同抽樣順序
sims = {"GBM": sim_gbm(N_MC, T_MC + WARM, rg_x)[:, WARM:], "Regime": sim_regime(N_MC, T_MC + WARM, rg_x)[:, WARM:]}
lr_h = np.log1p(Rb)
for q in (4, 13, 26):
    vh = vr(lr_h, q)[0]
    print(f"[E3] VR({q}) 歷史 {vh:.3f} | " + " | ".join(f"{k} 分位 {np.mean(vr(v, q) < vh):.1%}" for k, v in sims.items()))
a_h = ac1(lr_h)[0]
print(f"[E3] AC(1) 歷史 {a_h:.3f} | " + " | ".join(f"{k} 分位 {np.mean(ac1(v) < a_h):.1%}" for k, v in sims.items()))

# ---- E4 剔除 2026 夏季後的歷史績效(單一事件依賴) ----
cut = base < pd.Timestamp("2026-06-01"); rows = []
for nm, r in [("BuyHold", Rb), ("MA26", strat_r(26, COST)), ("MA40", strat_r(40, COST))]:
    for tag, msk in [("full", np.ones(Tb, bool)), ("ex_2026", cut)]:
        rr = r[msk]; mdd, pk, tr = dd_info(rr, base[msk])
        rows.append(dict(strat=nm, period=tag, CAGR=cagr_np(rr), MDD=mdd, Calmar=calmar_np(rr), peak=pk, trough=tr))
TABLES["X_leave_one_event"] = pd.DataFrame(rows); print("\n[E4]\n", TABLES["X_leave_one_event"].round(3).to_string(index=False))

# ---- E5 每個 L 的 MDD 由哪次空頭決定(驗證 F6 階梯斷崖) ----
rows = []
for k, L in enumerate(Lall):
    mdd, pk, tr = dd_info(strat_r(L, C0), base); rows.append(dict(L=int(L), calmar=cal[k], mdd=mdd, peak=pk, trough=tr))
mdd_ev = pd.DataFrame(rows); TABLES["X_mdd_event_by_L"] = mdd_ev
print("\n[E5] MDD 事件切換點:\n", mdd_ev[mdd_ev.trough != mdd_ev.trough.shift()].round(3).to_string(index=False))

# ---- E6 成本 0、間隔 1 的最佳 L(與 F6 星號、F6b 對照) ----
cal0 = np.array([calmar_np(strat_r(L, 0.0)) for L in Lall])
print(f"\n[E6] 成本 0:最佳 L={Lall[int(np.nanargmax(cal0))]},Calmar {np.nanmax(cal0):.3f};"
      f"L=12–20 範圍 {cal0[2:11].min():.3f}–{cal0[2:11].max():.3f}")

# ---- E7 延遲 0 對 1 的 CAGR(查核原說明書的 17.3% / 16.5%) ----
print(f"\n[E7] 買進持有 CAGR {GB:.2%}")
for lag in (0, 1):
    for L in (26, 40):
        print(f"  lag={lag} MA{L}: " + ", ".join(f"成本 {c:.1%} → CAGR {cagr_np(strat_r(L, c, lag=lag)):.2%}" for c in (0, 0.002, 0.004)))

# ---- E8 三次樣條局部斜率(E_smooth_orders FAIL 的佐證) ----
cv = E_smooth["cubic"].to_numpy(); hh = np.array(hs)
loc = np.diff(np.log(cv)) / np.diff(np.log(hh)); fin3 = np.polyfit(np.log(hh[1:]), np.log(cv[1:]), 1)[0]
print(f"\n[E8] 三次樣條局部斜率 {np.round(loc, 2).tolist()};只用最細三點擬合斜率 {fin3:.2f}(僅供說明,不改判定)")

# ---- E9 資料一致性:各年筆數、週末筆數、與前值相同筆數、小數位數 ----
yr = fund.index.year; same = fund.diff() == 0
dec = fund.map(lambda v: len(f"{v:.6f}".rstrip("0").split(".")[1]) if "." in f"{v:.6f}".rstrip("0") else 0)
dq = pd.DataFrame(dict(n_rows=fund.groupby(yr).size(), n_weekend=pd.Series(fund.index.weekday >= 5, index=fund.index).groupby(yr).sum(),
                       n_same_as_prev=same.groupby(yr).sum(), max_decimals=dec.groupby(yr).max()))
TABLES["X_data_quality_by_year"] = dq; print("\n[E9]\n", dq.to_string())

for k in ("X_leave_one_event", "X_mdd_event_by_L", "X_data_quality_by_year"):
    TABLES[k].to_csv(OUT / f"num_{k}.csv", index=(k == "X_data_quality_by_year"))

# (1) 剔除 GFC:均線優勢是否依賴 GFC(兩段報酬直接接續,僅供參考)
gfc = (base < pd.Timestamp("2007-07-01")) | (base > pd.Timestamp("2009-12-31"))
for nm, r in [("BuyHold", Rb), ("MA26", strat_r(26, COST)), ("MA40", strat_r(40, COST))]:
    rr = r[gfc]; print(f"{nm} ex_GFC: CAGR {cagr_np(rr):.3f}, MDD {dd_info(rr, base[gfc])[0]:.3f}, Calmar {calmar_np(rr):.3f}")

# (2) 更細的 h:確認三次樣條是否收斂到 4 階
g = lambda t: np.sin(2 * np.pi * t); xf_ = np.linspace(0, 1, 8001)
for h in (1/64, 1/128, 1/256):
    xs = np.linspace(0, 1, int(round(1 / h)) + 1)
    print(h, np.sqrt(np.mean((interpolate.CubicSpline(xs, g(xs))(xf_) - g(xf_)) ** 2)))

ev = TABLES["X_mdd_event_by_L"].copy()
yr = pd.to_datetime(ev["trough"]).dt.year
ev["event"] = np.select([yr <= 2003, yr <= 2009, yr == 2020],
                        ["2002–03 bear", "GFC 2007–09", "COVID 2020"], "Summer 2026")
ev_col = {"2002–03 bear": COL["purple"], "GFC 2007–09": COL["red"],
          "COVID 2020": COL["green"], "Summer 2026": COL["orange"]}

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(ev.L, ev.calmar, color=COL["grey"], lw=1, zorder=1)
for nm, g in ev.groupby("event"):
    ax.scatter(g.L, g.calmar, s=40, color=ev_col[nm], label=f"MDD set by {nm}", zorder=2)
ax.axhline(CB, color="black", ls="--", lw=1, label=f"buy & hold {CB:.2f}")
for L0 in (26.5, 51.5):
    ax.axvline(L0, color=COL["grey"], ls=":", lw=1)
ax.set_xlabel("MA length L (weeks)"); ax.set_ylabel("Calmar (cost 0.2%)")
ax.set_title("S8  Calmar cliffs occur where the MDD-defining crash switches", fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
save(fig, "S8_calmar_by_event")
